# LangGraph Human-in-the-Loop
## Content Publishing Pipeline

AI agents are powerful, but many real-world workflows require **human oversight** at critical decision points. LangGraph provides first-class support for pausing execution, waiting for human input, and resuming with that feedback.

### Three HITL Patterns

| Pattern | How It Works | Use Case |
|---------|-------------|----------|
| **Draft Review** | AI generates → Human reviews → AI revises | Content creation, report writing |
| **Tool Approval** | AI proposes tool calls → Human approves/denies → Execute | Dangerous operations (publish, delete, send) |
| **Collaborative Editing** | AI drafts → Human edits → AI refines → Repeat | Iterative content refinement |

### Key LangGraph APIs

```python
# Pause execution before a node
graph.compile(checkpointer=checkpointer, interrupt_before=["review_node"])

# Pause execution after a node  
graph.compile(checkpointer=checkpointer, interrupt_after=["draft_node"])

# Resume with human input
graph.invoke(Command(resume={"feedback": "approved"}), config=config)

# Inspect paused state
state = graph.get_state(config)
print(state.next)  # Which node will execute next
```

### What We're Building

A content publishing pipeline where:
- AI **researches** a topic and **drafts** an article
- A human **reviews** the draft and can approve, request edits, or reject
- If approved, the AI can propose **tool calls** (publish, email) that require human approval
- The system supports **iterative refinement** through multiple review rounds

In [ ]:
%pip install -q langgraph langchain langchain-openai langchain-community

In [ ]:
import os
import getpass
from typing import Annotated, TypedDict, Literal, Any

from pydantic import BaseModel, Field

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_openai import ChatOpenAI

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
print("Setup complete!")

---
# Part A: Draft Review Pipeline
## AI Generates, Human Reviews, AI Revises

This is the most common HITL pattern. The AI does the heavy lifting (research + writing), but a human reviews before anything is published.

### How Interrupts Work

```
researcher → drafter → [INTERRUPT] → human reviews → review_router → ...
                            │
                 Graph pauses here.
                 State is saved to checkpointer.
                 Human provides feedback.
                 Graph resumes from this point.
```

### The Checkpointer

LangGraph uses a **checkpointer** to save graph state when interrupted. `MemorySaver` stores state in memory (for development). In production, you'd use `SqliteSaver` or `PostgresSaver`.

```python
memory = MemorySaver()
app = graph.compile(checkpointer=memory, interrupt_before=["human_review"])
```

In [ ]:
class PublishingState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str
    research_notes: str
    draft_content: str
    human_feedback: str
    approval_status: str  # "pending", "approved", "needs_revision", "rejected"
    revision_count: int

print("PublishingState defined for draft review pipeline")

In [ ]:
def researcher(state: PublishingState) -> dict:
    """Research a topic and compile notes."""
    response = llm.invoke([
        SystemMessage(content="""You are a research assistant. Research the given topic and compile 
key points, facts, and interesting angles. Provide structured research notes 
that a writer can use to draft an article. Include 5-7 key points."""),
        HumanMessage(content=f"Research this topic: {state['topic']}"),
    ])
    
    return {
        "research_notes": response.content,
        "messages": [AIMessage(content=f"[Researcher] Research complete. Compiled notes on: {state['topic']}")],
    }

def drafter(state: PublishingState) -> dict:
    """Draft an article based on research notes."""
    revision_count = state.get("revision_count", 0)
    feedback = state.get("human_feedback", "")
    
    if revision_count > 0 and feedback:
        prompt = f"""You are a content writer. Revise the following draft based on human feedback.

PREVIOUS DRAFT:
{state.get('draft_content', '')}

HUMAN FEEDBACK:
{feedback}

RESEARCH NOTES:
{state.get('research_notes', '')}

Write an improved version addressing all feedback points."""
    else:
        prompt = f"""You are a content writer. Write a compelling article draft based on these research notes.

RESEARCH NOTES:
{state.get('research_notes', '')}

Write a well-structured article with:
- Engaging headline
- Strong opening paragraph
- 3-4 body sections with subheadings
- Conclusion with key takeaway

Keep it concise (300-400 words)."""
    
    response = llm.invoke([
        SystemMessage(content=prompt),
        HumanMessage(content=f"Topic: {state['topic']}"),
    ])
    
    return {
        "draft_content": response.content,
        "messages": [AIMessage(content=f"[Drafter] {'Revision' if revision_count > 0 else 'Initial draft'} #{revision_count + 1} complete.")],
        "approval_status": "pending",
    }

print("Researcher and drafter nodes defined")

In [ ]:
def human_review(state: PublishingState) -> dict:
    """Pause for human review using interrupt()."""
    # This is where the magic happens — interrupt() pauses the graph
    # and waits for human input via Command(resume=...)
    feedback = interrupt({
        "draft": state.get("draft_content", ""),
        "revision_number": state.get("revision_count", 0) + 1,
        "prompt": "Please review this draft. Respond with: 'approved', 'rejected', or provide revision feedback.",
    })
    
    # When resumed, feedback contains the human's response
    if isinstance(feedback, str):
        feedback_lower = feedback.lower().strip()
        if feedback_lower == "approved":
            return {
                "human_feedback": feedback,
                "approval_status": "approved",
                "messages": [AIMessage(content="[Human Review] Draft APPROVED!")],
            }
        elif feedback_lower == "rejected":
            return {
                "human_feedback": feedback,
                "approval_status": "rejected",
                "messages": [AIMessage(content="[Human Review] Draft REJECTED.")],
            }
        else:
            return {
                "human_feedback": feedback,
                "approval_status": "needs_revision",
                "revision_count": state.get("revision_count", 0) + 1,
                "messages": [AIMessage(content=f"[Human Review] Revision requested: {feedback[:100]}...")],
            }
    
    return {
        "human_feedback": str(feedback),
        "approval_status": "needs_revision",
        "revision_count": state.get("revision_count", 0) + 1,
    }

print("Human review node defined with interrupt()")

In [ ]:
def route_after_review(state: PublishingState) -> str:
    """Route based on human review decision."""
    status = state.get("approval_status", "pending")
    if status == "approved":
        return "publish"
    elif status == "rejected":
        return END
    else:  # needs_revision
        return "drafter"  # Back to revision

def publish_node(state: PublishingState) -> dict:
    """Simulate publishing the approved content."""
    return {
        "messages": [AIMessage(content=f"[Publisher] Article published successfully! Title: {state.get('draft_content', '')[:80]}...")],
        "approval_status": "published",
    }

# Build the graph
review_graph = StateGraph(PublishingState)

review_graph.add_node("researcher", researcher)
review_graph.add_node("drafter", drafter)
review_graph.add_node("human_review", human_review)
review_graph.add_node("publish", publish_node)

review_graph.add_edge(START, "researcher")
review_graph.add_edge("researcher", "drafter")
review_graph.add_edge("drafter", "human_review")
review_graph.add_conditional_edges("human_review", route_after_review, {
    "publish": "publish",
    "drafter": "drafter",
    END: END,
})
review_graph.add_edge("publish", END)

# Compile with checkpointer (required for interrupts)
memory = MemorySaver()
review_app = review_graph.compile(checkpointer=memory)
print("Draft review pipeline compiled with MemorySaver checkpointer!")

In [ ]:
from IPython.display import display, Image

try:
    display(Image(review_app.get_graph().draw_mermaid_png()))
except Exception:
    print(review_app.get_graph().draw_mermaid())

### Running the Draft Review Pipeline

When the graph hits `human_review`, it **pauses**. We can then:
1. **Inspect the state** to see the draft
2. **Resume with feedback** using `Command(resume=...)`

This simulates what would happen in a real application where a human reviews a draft in a UI.

In [ ]:
# Step 1: Start the pipeline — it will pause at human_review
config = {"configurable": {"thread_id": "review-1"}}

result = review_app.invoke(
    {
        "messages": [],
        "topic": "The Rise of AI Agents in Enterprise Software: How autonomous AI systems are transforming business operations",
        "research_notes": "",
        "draft_content": "",
        "human_feedback": "",
        "approval_status": "pending",
        "revision_count": 0,
    },
    config=config,
)

print("=" * 80)
print("STEP 1: Research + Draft (paused at human review)")
print("=" * 80)

# Check what state we're in
state = review_app.get_state(config)
print(f"\nGraph paused. Next node(s): {state.next}")
print(f"\nDraft content preview:")
print(f"{result.get('draft_content', 'No draft yet')[:500]}...")

In [ ]:
# Step 2: Human requests revisions
print("=" * 80)
print("STEP 2: Human requests revision")
print("=" * 80)

result2 = review_app.invoke(
    Command(resume="Please add more specific examples of companies using AI agents. Also make the tone more conversational and less formal. Add a section about potential risks."),
    config=config,
)

# Check state again
state = review_app.get_state(config)
print(f"\nGraph paused again. Next node(s): {state.next}")
print(f"Revision count: {result2.get('revision_count', 0)}")
print(f"\nRevised draft preview:")
print(f"{result2.get('draft_content', 'No draft')[:500]}...")

In [ ]:
# Step 3: Human approves the revision
print("=" * 80)
print("STEP 3: Human approves")
print("=" * 80)

result3 = review_app.invoke(
    Command(resume="approved"),
    config=config,
)

print(f"\nFinal status: {result3.get('approval_status')}")
print(f"Total revisions: {result3.get('revision_count', 0)}")

# Show all messages
for msg in result3["messages"]:
    if isinstance(msg, AIMessage):
        print(f"\n  {msg.content[:200]}")

---
# Part B: Tool Execution Approval
## Safe vs Dangerous Tool Classification

Some tool calls are safe (reading data, searching), but others have real-world consequences (publishing, sending emails, deleting records). This pattern lets a human review dangerous tool calls before execution.

### The Pattern

```
agent → (has tool calls?) →┬→ safe tools → auto-execute → agent
                            └→ dangerous tools → [INTERRUPT] → human approves → execute or skip
```

### Key Insight

We don't interrupt for EVERY tool call — only for tools classified as **dangerous**. This keeps the workflow smooth for routine operations while adding guardrails for high-impact actions.

In [ ]:
from langchain_core.tools import tool

class ToolApprovalState(TypedDict):
    messages: Annotated[list, add_messages]
    pending_tool_calls: list  # Tool calls waiting for approval
    tool_approval_log: list  # Record of approvals/denials

# Define safe and dangerous tools
@tool
def search_articles(query: str) -> str:
    """Search for existing articles on a topic. (SAFE)"""
    return f"Found 3 relevant articles about '{query}': 1) Industry overview, 2) Recent trends, 3) Expert analysis"

@tool
def check_seo_score(title: str) -> str:
    """Check the SEO score of an article title. (SAFE)"""
    return f"SEO score for '{title}': 78/100. Suggestions: Add target keyword, keep under 60 chars."

@tool
def publish_article(title: str, content: str) -> str:
    """Publish an article to the website. (DANGEROUS - requires approval)"""
    return f"Article '{title}' published successfully to production website."

@tool
def send_newsletter(subject: str, recipient_list: str) -> str:
    """Send a newsletter to subscribers. (DANGEROUS - requires approval)"""
    return f"Newsletter '{subject}' sent to {recipient_list} subscribers."

@tool
def delete_article(article_id: str) -> str:
    """Delete an article from the website. (DANGEROUS - requires approval)"""
    return f"Article {article_id} permanently deleted."

# Classify tools
SAFE_TOOLS = {"search_articles", "check_seo_score"}
DANGEROUS_TOOLS = {"publish_article", "send_newsletter", "delete_article"}

all_tools = [search_articles, check_seo_score, publish_article, send_newsletter, delete_article]
agent_llm = llm.bind_tools(all_tools)

print("Tools defined:")
print(f"  Safe: {SAFE_TOOLS}")
print(f"  Dangerous: {DANGEROUS_TOOLS}")

In [ ]:
def content_agent(state: ToolApprovalState) -> dict:
    """AI agent that may call safe or dangerous tools."""
    response = agent_llm.invoke([
        SystemMessage(content="""You are a content management agent. You can:
- search_articles: Search for existing content (safe)
- check_seo_score: Check title SEO (safe)
- publish_article: Publish to website (requires human approval)
- send_newsletter: Send to subscribers (requires human approval)
- delete_article: Remove content (requires human approval)

Complete the user's request using appropriate tools."""),
        *state["messages"],
    ])
    
    return {"messages": [response]}

def tool_classifier(state: ToolApprovalState) -> str:
    """Classify whether pending tool calls are safe or need approval."""
    last_message = state["messages"][-1]
    
    if not hasattr(last_message, "tool_calls") or not last_message.tool_calls:
        return "done"
    
    # Check if any tool call is dangerous
    for tc in last_message.tool_calls:
        if tc["name"] in DANGEROUS_TOOLS:
            return "needs_approval"
    
    return "safe_execute"

def safe_executor(state: ToolApprovalState) -> dict:
    """Auto-execute safe tools without human intervention."""
    from langgraph.prebuilt import ToolNode
    safe_tool_objects = [t for t in all_tools if t.name in SAFE_TOOLS]
    tool_node = ToolNode(safe_tool_objects)
    result = tool_node.invoke({"messages": state["messages"]})
    return {"messages": result["messages"]}

def approval_gate(state: ToolApprovalState) -> dict:
    """Pause for human approval of dangerous tool calls."""
    last_message = state["messages"][-1]
    dangerous_calls = [tc for tc in last_message.tool_calls if tc["name"] in DANGEROUS_TOOLS]
    
    # Interrupt and present dangerous tool calls for review
    decision = interrupt({
        "dangerous_tool_calls": [
            {"tool": tc["name"], "args": tc["args"]} 
            for tc in dangerous_calls
        ],
        "prompt": "Review these tool calls. Respond 'approve' to execute all, 'deny' to skip, or specify which to approve.",
    })
    
    if isinstance(decision, str) and decision.lower().strip() == "approve":
        # Execute all dangerous tools
        from langgraph.prebuilt import ToolNode
        dangerous_tool_objects = [t for t in all_tools if t.name in DANGEROUS_TOOLS]
        tool_node = ToolNode(dangerous_tool_objects)
        result = tool_node.invoke({"messages": state["messages"]})
        return {
            "messages": result["messages"],
            "tool_approval_log": state.get("tool_approval_log", []) + [
                {"tools": [tc["name"] for tc in dangerous_calls], "decision": "approved"}
            ],
        }
    else:
        # Skip execution
        from langchain_core.messages import ToolMessage
        skip_messages = [
            ToolMessage(content=f"Tool call DENIED by human reviewer.", tool_call_id=tc["id"])
            for tc in dangerous_calls
        ]
        return {
            "messages": skip_messages,
            "tool_approval_log": state.get("tool_approval_log", []) + [
                {"tools": [tc["name"] for tc in dangerous_calls], "decision": "denied"}
            ],
        }

print("Tool approval nodes defined: content_agent, safe_executor, approval_gate")

In [ ]:
tool_graph = StateGraph(ToolApprovalState)

tool_graph.add_node("agent", content_agent)
tool_graph.add_node("safe_execute", safe_executor)
tool_graph.add_node("approval_gate", approval_gate)

tool_graph.add_edge(START, "agent")
tool_graph.add_conditional_edges("agent", tool_classifier, {
    "safe_execute": "safe_execute",
    "needs_approval": "approval_gate",
    "done": END,
})
tool_graph.add_edge("safe_execute", "agent")  # Back to agent after safe execution
tool_graph.add_edge("approval_gate", "agent")  # Back to agent after approval decision

tool_memory = MemorySaver()
tool_app = tool_graph.compile(checkpointer=tool_memory)
print("Tool approval graph compiled!")

In [ ]:
try:
    display(Image(tool_app.get_graph().draw_mermaid_png()))
except Exception:
    print(tool_app.get_graph().draw_mermaid())

In [ ]:
# Scenario: Agent uses only safe tools (no interruption)
config_safe = {"configurable": {"thread_id": "tools-safe-1"}}

result_safe = tool_app.invoke(
    {
        "messages": [HumanMessage(content="Search for articles about AI agents and check the SEO score for the title 'AI Agents Are Transforming Enterprise Software'")],
        "pending_tool_calls": [],
        "tool_approval_log": [],
    },
    config=config_safe,
)

print("=" * 80)
print("SAFE TOOLS SCENARIO (no interruption)")
print("=" * 80)
for msg in result_safe["messages"]:
    if isinstance(msg, AIMessage) and msg.content:
        print(f"\n  [Agent] {msg.content[:300]}")
    elif isinstance(msg, HumanMessage):
        print(f"\n  [Human] {msg.content[:200]}")

print(f"\nTool approval log: {result_safe.get('tool_approval_log', [])}")

In [ ]:
# Scenario: Agent needs to publish (dangerous — will interrupt)
config_dangerous = {"configurable": {"thread_id": "tools-dangerous-1"}}

result_d1 = tool_app.invoke(
    {
        "messages": [HumanMessage(content="Publish an article with title 'AI Agents in 2024' and content 'AI agents are revolutionizing how businesses operate...' to the website.")],
        "pending_tool_calls": [],
        "tool_approval_log": [],
    },
    config=config_dangerous,
)

print("=" * 80)
print("DANGEROUS TOOLS SCENARIO — Step 1: Agent proposes, graph pauses")
print("=" * 80)

state = tool_app.get_state(config_dangerous)
print(f"Graph paused. Next node(s): {state.next}")
print(f"\nPending approval for dangerous tool calls...")

In [ ]:
# Human approves the publication
print("=" * 80)
print("DANGEROUS TOOLS SCENARIO — Step 2: Human approves")
print("=" * 80)

result_d2 = tool_app.invoke(
    Command(resume="approve"),
    config=config_dangerous,
)

for msg in result_d2["messages"]:
    if isinstance(msg, AIMessage) and msg.content:
        print(f"\n  [Agent] {msg.content[:300]}")

print(f"\nTool approval log: {result_d2.get('tool_approval_log', [])}")

---
# Part C: Collaborative Editing
## Turn-Taking Between AI and Human

This pattern creates a **back-and-forth** conversation where AI and human take turns refining content. Unlike the review pipeline (where the human just approves/rejects), here the human actively **edits** the content.

### The Pattern

```
ai_draft → [INTERRUPT: human edits] → ai_refine → [INTERRUPT: human edits] → ... → finalize
```

Each cycle:
1. AI generates/refines content
2. Human reviews and provides edits, suggestions, or says "done"
3. AI incorporates feedback and refines
4. Repeat until human is satisfied

In [ ]:
class CollabState(TypedDict):
    messages: Annotated[list, add_messages]
    topic: str
    current_draft: str
    edit_history: list  # Track all versions
    turn_count: int
    status: str  # "drafting", "editing", "finalized"

def ai_draft(state: CollabState) -> dict:
    """AI creates or refines a draft based on human edits."""
    turn = state.get("turn_count", 0)
    current = state.get("current_draft", "")
    
    if turn == 0:
        # Initial draft
        response = llm.invoke([
            SystemMessage(content="""Write a concise first draft (200-300 words) on the given topic. 
Include a title, 2-3 sections, and a conclusion. Keep it as a starting point for collaborative editing."""),
            HumanMessage(content=f"Topic: {state['topic']}"),
        ])
    else:
        # Refine based on history
        response = llm.invoke([
            SystemMessage(content=f"""Refine this draft based on the latest feedback from the human editor.
Incorporate their changes while maintaining coherent flow.

CURRENT DRAFT:
{current}"""),
            *state["messages"][-3:],  # Recent context
        ])
    
    return {
        "current_draft": response.content,
        "messages": [AIMessage(content=f"[AI Draft - Turn {turn + 1}] Here's the {'initial' if turn == 0 else 'revised'} draft:\n\n{response.content}")],
        "status": "editing",
        "turn_count": turn + 1,
        "edit_history": state.get("edit_history", []) + [{"turn": turn + 1, "author": "AI", "content": response.content[:200]}],
    }

def human_edit(state: CollabState) -> dict:
    """Pause for human to provide edits or approve."""
    feedback = interrupt({
        "current_draft": state.get("current_draft", ""),
        "turn": state.get("turn_count", 0),
        "prompt": "Review the draft. Provide specific edits/suggestions, or say 'done' to finalize.",
    })
    
    if isinstance(feedback, str) and feedback.lower().strip() == "done":
        return {
            "status": "finalized",
            "messages": [AIMessage(content="[Human Editor] Content finalized!")],
            "edit_history": state.get("edit_history", []) + [{"turn": state.get("turn_count", 0), "author": "Human", "content": "FINALIZED"}],
        }
    
    return {
        "messages": [HumanMessage(content=f"Please make these edits: {feedback}")],
        "status": "drafting",
        "edit_history": state.get("edit_history", []) + [{"turn": state.get("turn_count", 0), "author": "Human", "content": str(feedback)[:200]}],
    }

def finalize(state: CollabState) -> dict:
    """Finalize the collaboratively edited content."""
    return {
        "messages": [AIMessage(content=f"[Finalized] Content complete after {state.get('turn_count', 0)} rounds of editing.\n\nFINAL VERSION:\n{state.get('current_draft', '')}")],
        "status": "finalized",
    }

def route_collab(state: CollabState) -> str:
    if state.get("status") == "finalized":
        return "finalize"
    return "ai_draft"

print("Collaborative editing nodes defined: ai_draft, human_edit, finalize")

In [ ]:
collab_graph = StateGraph(CollabState)

collab_graph.add_node("ai_draft", ai_draft)
collab_graph.add_node("human_edit", human_edit)
collab_graph.add_node("finalize", finalize)

collab_graph.add_edge(START, "ai_draft")
collab_graph.add_edge("ai_draft", "human_edit")
collab_graph.add_conditional_edges("human_edit", route_collab, {
    "ai_draft": "ai_draft",
    "finalize": "finalize",
})
collab_graph.add_edge("finalize", END)

collab_memory = MemorySaver()
collab_app = collab_graph.compile(checkpointer=collab_memory)
print("Collaborative editing graph compiled!")

In [ ]:
try:
    display(Image(collab_app.get_graph().draw_mermaid_png()))
except Exception:
    print(collab_app.get_graph().draw_mermaid())

In [ ]:
collab_config = {"configurable": {"thread_id": "collab-1"}}

# Turn 1: AI drafts
collab_r1 = collab_app.invoke(
    {
        "messages": [],
        "topic": "Why Every Developer Should Understand AI Agents in 2024",
        "current_draft": "",
        "edit_history": [],
        "turn_count": 0,
        "status": "drafting",
    },
    config=collab_config,
)

print("=" * 80)
print("COLLABORATIVE EDITING — Turn 1: AI Initial Draft")
print("=" * 80)
print(f"\n{collab_r1.get('current_draft', '')[:600]}...")

state = collab_app.get_state(collab_config)
print(f"\nPaused at: {state.next}")

In [ ]:
# Turn 2: Human provides edits
print("=" * 80)
print("COLLABORATIVE EDITING — Turn 2: Human Edits")
print("=" * 80)

collab_r2 = collab_app.invoke(
    Command(resume="Make the opening more attention-grabbing with a bold prediction. Also add a section about specific tools developers should learn (LangGraph, CrewAI). Make the conclusion more actionable with 3 concrete next steps."),
    config=collab_config,
)

print(f"\nRevised draft preview:")
print(f"{collab_r2.get('current_draft', '')[:600]}...")

state = collab_app.get_state(collab_config)
print(f"\nPaused at: {state.next}")

In [ ]:
# Turn 3: Human approves
print("=" * 80)
print("COLLABORATIVE EDITING — Turn 3: Human Finalizes")
print("=" * 80)

collab_r3 = collab_app.invoke(
    Command(resume="done"),
    config=collab_config,
)

print(f"Status: {collab_r3.get('status')}")
print(f"Total turns: {collab_r3.get('turn_count')}")

print(f"\nEdit History:")
for entry in collab_r3.get("edit_history", []):
    print(f"  Turn {entry['turn']} ({entry['author']}): {entry['content'][:80]}...")

## State Inspection APIs

LangGraph provides powerful APIs for inspecting and modifying the state of a paused graph. These are essential for building HITL UIs.

```python
# Get current state (including values and next nodes)
state = graph.get_state(config)
print(state.values)      # Current state values
print(state.next)        # Next node(s) to execute
print(state.config)      # Thread configuration

# Get state history (all checkpoints)
for state in graph.get_state_history(config):
    print(state.values["approval_status"])
```

In [ ]:
# Inspect the full state history of our review pipeline
print("=" * 80)
print("STATE HISTORY: Draft Review Pipeline")
print("=" * 80)

history = list(review_app.get_state_history(config))
print(f"\nTotal checkpoints: {len(history)}")

for i, state_snapshot in enumerate(reversed(history)):
    status = state_snapshot.values.get("approval_status", "N/A")
    revision = state_snapshot.values.get("revision_count", 0)
    next_nodes = state_snapshot.next
    draft_preview = state_snapshot.values.get("draft_content", "")[:60]
    
    print(f"\n  Checkpoint {i + 1}:")
    print(f"    Status: {status}")
    print(f"    Revision: {revision}")
    print(f"    Next: {next_nodes}")
    if draft_preview:
        print(f"    Draft: {draft_preview}...")

## Key Takeaways

### What We Built

Three human-in-the-loop patterns:
1. **Draft Review** — AI generates, human approves/revises, with iterative feedback loops
2. **Tool Approval** — Safe tools auto-execute, dangerous tools require human approval
3. **Collaborative Editing** — Turn-taking between AI and human for iterative refinement

### Essential HITL APIs

| API | Purpose |
|-----|--------|
| `interrupt(value)` | Pause graph, present data to human, wait for response |
| `Command(resume=value)` | Resume a paused graph with human input |
| `MemorySaver()` | In-memory checkpointer for state persistence |
| `graph.get_state(config)` | Inspect current state of a paused graph |
| `graph.get_state_history(config)` | View all state checkpoints |
| `graph.update_state(config, values)` | Directly modify graph state |

### Design Principles

1. **Classify tool safety** — Not every action needs human approval. Define clear safe/dangerous boundaries.
2. **Use structured interrupts** — Pass context to humans (drafts, tool calls) so they can make informed decisions.
3. **Support iteration** — Humans rarely approve on first review. Design for multiple revision rounds.
4. **Track history** — The checkpointer gives you time-travel debugging and audit trails for free.
5. **Thread IDs matter** — Each conversation/workflow needs a unique thread ID for state isolation.

### When to Use HITL

| Scenario | HITL Pattern |
|----------|-------------|
| Content must be reviewed before publishing | Draft Review |
| Some operations have real-world consequences | Tool Approval |
| Output quality requires domain expertise | Collaborative Editing |
| Regulatory/compliance requirements | Draft Review + Tool Approval |
| Creative work (marketing copy, design briefs) | Collaborative Editing |

### Production Considerations

- **Replace `MemorySaver`** with `PostgresSaver` or `SqliteSaver` for persistence
- **Build a UI** that reads `graph.get_state()` and calls `graph.invoke(Command(resume=...))` 
- **Add timeouts** — What happens if the human never responds?
- **Support delegation** — Allow one human to pass the review to another
- **Audit logging** — Track who approved what and when